# Analysis 5: Loss Function Ablation (Weighted CE + Focal Loss)

## Purpose

Method4 (CE+Dice) achieved the best Mean Dice of 0.9522. This ablation study investigates whether
**class-imbalance-aware loss functions** can further improve performance, especially for rare classes.

## 6-Class Distribution (Severe Imbalance)

| Class | Pixel % | Inverse Freq Weight (normalized) |
|-------|---------|----------------------------------|
| Background (0) | 86.49% | 0.0116 |
| Conjunctiva (1) | 5.70% | 0.1755 |
| Iris visible (2) | 5.43% | 0.1843 |
| Iris occluded (3) | 2.08% | 0.4808 |
| Pupil visible (4) | 0.87% | 1.1494 |
| Pupil occluded (5) | 0.03% | 33.3333 |

## Loss Variants Compared

| Variant | Loss Function | Hypothesis |
|---------|--------------|------------|
| Baseline (Method4) | CE + 0.5*Dice | Reference |
| Weighted CE + Dice | WeightedCE(w=inv_freq) + 0.5*Dice | Explicit class rebalancing improves rare-class Dice |
| Focal + Dice | FocalLoss(gamma=2) + 0.5*Dice | Hard-example mining improves boundary precision |

In [ ]:
import os
import json
import random
import time
from pathlib import Path
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchvision
from tqdm.auto import tqdm
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# ----- GPU check -----
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    raise SystemError('GPU not available')

# ----- Reproducibility -----
GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
torch.cuda.manual_seed_all(GLOBAL_SEED)
torch.backends.cudnn.benchmark = True

# ----- Paths / Hyperparameters -----
IMAGES_DIR    = Path('Images/images')
LABEL_SEG_DIR = Path('Images/labels_seg')
LABEL_OBB_DIR = Path('Images/labels_obb')
ABLATION_MODEL_DIR = Path('model/cv_300ep_ablation_loss')
ABLATION_MODEL_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_HEIGHT = 512
IMAGE_WIDTH  = 512
BATCH_SIZE   = 16
NUM_EPOCHS   = 300
LEARNING_RATE = 1e-3
WEIGHT_DECAY  = 1e-4
EARLY_STOP_PATIENCE = 30
NUM_FOLDS = 5
NUM_WORKERS = 0
PIN_MEMORY = True

LOSS_VARIANTS = ['weighted_ce_dice', 'focal_dice']

# fold indices
with open('fold_indices.json', 'r') as f:
    fold_indices = json.load(f)

# image list
df = pd.read_csv('image_metadata.csv')
image_paths = [IMAGES_DIR / row['filename'] for _, row in df.iterrows()]

print(f'Images: {len(image_paths)}')
print(f'Model dir: {ABLATION_MODEL_DIR}')
print(f'Loss variants: {LOSS_VARIANTS}')

In [ ]:
# ===== Class weights from pixel distribution analysis =====
# From analysis 1: 6-class pixel distribution across all 1992 images
CLASS_PIXEL_FRACTIONS = np.array([
    0.8649,  # background
    0.0570,  # conjunctiva
    0.0543,  # iris_vis
    0.0208,  # iris_occ
    0.0087,  # pupil_vis
    0.0003,  # pupil_occ  (cap at 0.0003 to avoid extreme weight)
])

# Inverse frequency weights
raw_weights = 1.0 / CLASS_PIXEL_FRACTIONS

# Normalize so mean = 1.0 (preserves loss magnitude for same lr/dice_weight)
ce_weights = raw_weights / raw_weights.mean()
ce_weights_tensor = torch.tensor(ce_weights, dtype=torch.float32)

print('Class weights (normalized mean=1.0):')
class_names = ['background', 'conjunctiva', 'iris_vis', 'iris_occ', 'pupil_vis', 'pupil_occ']
for name, frac, w in zip(class_names, CLASS_PIXEL_FRACTIONS, ce_weights):
    print(f'  {name:15s}: pixel_frac={frac:.4f}, weight={w:.4f}')
print(f'  Mean weight: {ce_weights.mean():.4f}')

In [ ]:
# ===== Dataset and Model definitions (from crossvalidation.ipynb) =====

# sixcls.png BGR to class ID mapping
SIXCLS_BGR_TO_ID = {
    (0, 0, 0): 0,         # background
    (255, 0, 0): 1,       # conjunctiva (blue in BGR)
    (0, 255, 0): 2,       # iris_vis (green)
    (0, 0, 255): 3,       # iris_occ (red in BGR)
    (0, 255, 255): 4,     # pupil_vis (yellow in BGR)
    (255, 0, 255): 5,     # pupil_occ (magenta in BGR)
}

def _resize_mask(mask, H=IMAGE_HEIGHT, W=IMAGE_WIDTH):
    if mask is None:
        return np.zeros((H, W), dtype=np.uint8)
    return cv2.resize(mask, (W, H), interpolation=cv2.INTER_NEAREST)

def convert_sixcls_to_labels(sixcls_img):
    H, W = sixcls_img.shape[:2]
    labels = np.zeros((H, W), dtype=np.uint8)
    for bgr_color, class_id in SIXCLS_BGR_TO_ID.items():
        mask = np.all(sixcls_img == bgr_color, axis=2)
        labels[mask] = class_id
    return labels

class EyeSegmentationDataset(Dataset):
    def __init__(self, image_paths, label_seg_dir, label_obb_dir, transform=True):
        self.image_paths = image_paths
        self.label_seg_dir = Path(label_seg_dir)
        self.label_obb_dir = Path(label_obb_dir)
        self.transform = transform

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        p = self.image_paths[idx]
        img = cv2.imread(str(p))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMAGE_WIDTH, IMAGE_HEIGHT), interpolation=cv2.INTER_LINEAR)

        stem = p.stem
        mask_lid   = cv2.imread(str(self.label_seg_dir / f'{stem}_mask_lid.png'),   0)
        mask_iris  = cv2.imread(str(self.label_obb_dir / f'{stem}_mask_iris.png'),  0)
        mask_pupil = cv2.imread(str(self.label_obb_dir / f'{stem}_mask_pupil.png'), 0)

        sixcls_path = self.label_seg_dir / f'{stem}_sixcls.png'
        sixcls_img = cv2.imread(str(sixcls_path))
        if sixcls_img is None:
            gt_sixcls = np.zeros((IMAGE_HEIGHT, IMAGE_WIDTH), dtype=np.uint8)
        else:
            sixcls_img = cv2.resize(sixcls_img, (IMAGE_WIDTH, IMAGE_HEIGHT),
                                    interpolation=cv2.INTER_NEAREST)
            gt_sixcls = convert_sixcls_to_labels(sixcls_img)

        mask_lid   = _resize_mask(mask_lid)
        mask_iris  = _resize_mask(mask_iris)
        mask_pupil = _resize_mask(mask_pupil)

        img_t = torch.from_numpy(img).float().permute(2,0,1) / 255.0
        mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
        std  = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
        img_t = (img_t - mean) / std

        return dict(
            image = img_t,
            mask_lid   = torch.from_numpy(mask_lid).long(),
            mask_iris  = torch.from_numpy(mask_iris).long(),
            mask_pupil = torch.from_numpy(mask_pupil).long(),
            gt_sixcls  = torch.from_numpy(gt_sixcls).long(),
            filename = p.name
        )

# ===== Model =====
class UNetEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = torchvision.models.vgg16_bn(weights='DEFAULT')
        self.features = vgg.features

    def forward(self, x):
        feats = {}
        for i, layer in enumerate(self.features):
            x = layer(x)
            if i == 5:  feats['0'] = x
            if i == 12: feats['1'] = x
            if i == 22: feats['2'] = x
            if i == 32: feats['3'] = x
        feats['4'] = x
        return feats

class UNetDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.dec4 = self._blk(512+512, 256)
        self.dec3 = self._blk(256+256, 128)
        self.dec2 = self._blk(128+128, 64)
        self.dec1 = self._blk(64+64,   64)

    def _blk(self, c_in, c_out):
        return nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
            nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU()
        )

    def forward(self, f):
        x = f['4']
        x = F.interpolate(x, size=f['3'].shape[-2:], mode='bilinear', align_corners=False)
        x = self.dec4(torch.cat([x, f['3']], 1))
        x = F.interpolate(x, size=f['2'].shape[-2:], mode='bilinear', align_corners=False)
        x = self.dec3(torch.cat([x, f['2']], 1))
        x = F.interpolate(x, size=f['1'].shape[-2:], mode='bilinear', align_corners=False)
        x = self.dec2(torch.cat([x, f['1']], 1))
        x = F.interpolate(x, size=f['0'].shape[-2:], mode='bilinear', align_corners=False)
        x = self.dec1(torch.cat([x, f['0']], 1))
        x = F.interpolate(x, size=(IMAGE_HEIGHT, IMAGE_WIDTH), mode='bilinear', align_corners=False)
        return x

class UNetMethod4(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = UNetEncoder()
        self.decoder = UNetDecoder()
        self.head_seg6 = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 6, 1)
        )

    def forward(self, x):
        f = self.encoder(x)
        d = self.decoder(f)
        return dict(five_class_seg = self.head_seg6(d))

# ===== Loss utilities =====
def dice_coeff(pred, tgt, smooth=1e-5):
    pred_f = pred.reshape(-1)
    tgt_f  = tgt.reshape(-1)
    inter  = (pred_f * tgt_f).sum()
    union  = pred_f.sum() + tgt_f.sum()
    return (2*inter + smooth) / (union + smooth)

def multi_class_dice_loss(prob, target, num_classes=6):
    scores = []
    for c in range(num_classes):
        pc = prob[:, c, :, :]
        tc = (target == c).float()
        if tc.sum() == 0: continue
        scores.append(dice_coeff(pc, tc))
    return prob.new_tensor(0.0) if len(scores)==0 else 1.0 - torch.mean(torch.stack(scores))

def build_sixclass_target(batch_masks, device):
    lid   = batch_masks['mask_lid'].to(device)   > 0
    iris  = batch_masks['mask_iris'].to(device)  > 0
    pupil = batch_masks['mask_pupil'].to(device) > 0
    B,H,W = lid.shape
    tgt = torch.zeros((B,H,W), dtype=torch.long, device=device)
    tgt[lid  & ~iris & ~pupil] = 1
    tgt[lid  &  iris & ~pupil] = 2
    tgt[~lid &  iris & ~pupil] = 3
    tgt[lid  &  iris &  pupil] = 4
    tgt[~lid &  iris &  pupil] = 5
    return tgt

print('Model and dataset definitions loaded.')

In [ ]:
# ===== Loss Function Definitions =====

class LossFunction4_Baseline(nn.Module):
    """Baseline: CE + 0.5*Dice (same as Method4)"""
    def __init__(self, dice_weight=0.5):
        super().__init__()
        self.ce_loss = nn.CrossEntropyLoss()
        self.dice_weight = dice_weight

    def forward(self, pred, target):
        logits = pred['five_class_seg']
        six_tgt = target['gt_sixcls'].to(logits.device)
        H, W = six_tgt.shape[-2:]
        if logits.shape[-2:] != (H, W):
            logits = F.interpolate(logits, size=(H, W), mode='bilinear', align_corners=False)
        loss = self.ce_loss(logits, six_tgt)
        prob = F.softmax(logits, dim=1)
        loss = loss + self.dice_weight * multi_class_dice_loss(prob, six_tgt, num_classes=6)
        return loss


class LossFunction4_WeightedCE(nn.Module):
    """Weighted CE + 0.5*Dice: class-imbalance-aware CE with inverse frequency weights"""
    def __init__(self, class_weights, dice_weight=0.5):
        super().__init__()
        self.register_buffer('class_weights', class_weights)
        self.dice_weight = dice_weight

    def forward(self, pred, target):
        logits = pred['five_class_seg']
        six_tgt = target['gt_sixcls'].to(logits.device)
        H, W = six_tgt.shape[-2:]
        if logits.shape[-2:] != (H, W):
            logits = F.interpolate(logits, size=(H, W), mode='bilinear', align_corners=False)
        # Weighted CE with device-matched weights
        loss = F.cross_entropy(logits, six_tgt, weight=self.class_weights.to(logits.device))
        prob = F.softmax(logits, dim=1)
        loss = loss + self.dice_weight * multi_class_dice_loss(prob, six_tgt, num_classes=6)
        return loss


class FocalLoss(nn.Module):
    """Focal Loss (Lin et al., 2017): -alpha * (1-p_t)^gamma * log(p_t)
    
    Numerically stable implementation using log_softmax + gather.
    No alpha weighting (gamma=2.0 only) to avoid over-weighting rare classes."""
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, targets):
        # logits: (B, C, H, W), targets: (B, H, W) long
        log_p = F.log_softmax(logits, dim=1)  # (B, C, H, W)
        # gather log probability of true class
        targets_expanded = targets.unsqueeze(1)  # (B, 1, H, W)
        log_pt = log_p.gather(1, targets_expanded).squeeze(1)  # (B, H, W)
        pt = log_pt.exp()  # probability of true class
        focal_weight = (1.0 - pt) ** self.gamma
        loss = -focal_weight * log_pt
        return loss.mean()


class LossFunction4_FocalDice(nn.Module):
    """Focal Loss + 0.5*Dice: hard-example-focused loss"""
    def __init__(self, gamma=2.0, dice_weight=0.5):
        super().__init__()
        self.focal = FocalLoss(gamma=gamma)
        self.dice_weight = dice_weight

    def forward(self, pred, target):
        logits = pred['five_class_seg']
        six_tgt = target['gt_sixcls'].to(logits.device)
        H, W = six_tgt.shape[-2:]
        if logits.shape[-2:] != (H, W):
            logits = F.interpolate(logits, size=(H, W), mode='bilinear', align_corners=False)
        loss = self.focal(logits, six_tgt)
        prob = F.softmax(logits, dim=1)
        loss = loss + self.dice_weight * multi_class_dice_loss(prob, six_tgt, num_classes=6)
        return loss


def create_criterion(variant_name):
    """Factory function to create loss criterion by variant name"""
    if variant_name == 'weighted_ce_dice':
        return LossFunction4_WeightedCE(ce_weights_tensor)
    elif variant_name == 'focal_dice':
        return LossFunction4_FocalDice(gamma=2.0)
    else:
        raise ValueError(f'Unknown variant: {variant_name}')

# Quick test
print('Loss functions defined.')
for name in LOSS_VARIANTS:
    c = create_criterion(name)
    print(f'  {name}: {c.__class__.__name__}')

In [ ]:
# ===== Training loop =====

def train_epoch(model, loader, criterion, optimizer, scaler, device, show_progress=True):
    model.train()
    total_loss = 0.0
    iterator = tqdm(loader, desc='Train', leave=False) if show_progress else loader
    for batch in iterator:
        image = batch['image'].to(device)
        optimizer.zero_grad(set_to_none=True)
        with autocast():
            out = model(image)
            target = {k:v.to(device) for k,v in batch.items()
                      if k.startswith('mask_') or k == 'gt_sixcls'}
            loss = criterion(out, target)
        if not torch.isfinite(loss):
            continue
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        if show_progress:
            iterator.set_postfix(loss=f'{loss.item():.4f}')
    return total_loss / max(1, len(loader))


@torch.no_grad()
def validate_epoch(model, loader, criterion, device, show_progress=True):
    model.eval()
    total_loss = 0.0
    iterator = tqdm(loader, desc='Valid', leave=False) if show_progress else loader
    for batch in iterator:
        image = batch['image'].to(device)
        with autocast():
            out = model(image)
            target = {k:v.to(device) for k,v in batch.items()
                      if k.startswith('mask_') or k == 'gt_sixcls'}
            loss = criterion(out, target)
        total_loss += loss.item()
        if show_progress:
            iterator.set_postfix(loss=f'{loss.item():.4f}')
    return total_loss / max(1, len(loader))


def run_ablation_fold(model, loader_tr, loader_va, criterion, optimizer, scaler,
                      device, variant_name, fold_idx):
    """Train one fold with early stopping and ETA display"""
    best = float('inf')
    patience = 0
    save_path = ABLATION_MODEL_DIR / f'{variant_name}_fold{fold_idx}_best.pth'
    epoch_times = []
    start_all = time.perf_counter()

    def fmt(t):
        m, s = divmod(int(t), 60)
        h, m = divmod(m, 60)
        return f'{h:02d}:{m:02d}:{s:02d}'

    for ep in range(1, NUM_EPOCHS + 1):
        start_ep = time.perf_counter()
        tr_loss = train_epoch(model, loader_tr, criterion, optimizer, scaler, device)
        va_loss = validate_epoch(model, loader_va, criterion, device)
        ep_time = time.perf_counter() - start_ep
        epoch_times.append(ep_time)
        avg_ep = np.mean(epoch_times)
        remaining = (NUM_EPOCHS - ep) * avg_ep

        print(f'[{variant_name}-F{fold_idx}] Epoch {ep:03d}/{NUM_EPOCHS} | '
              f'Train {tr_loss:.4f} | Val {va_loss:.4f} | '
              f'Time {fmt(ep_time)} | ETA {fmt(remaining)}')

        if va_loss < best:
            best = va_loss
            patience = 0
            torch.save({
                'model': model.state_dict(),
                'val_loss': best,
                'epoch': ep,
                'fold': fold_idx,
                'variant': variant_name,
            }, save_path)
            print(f'  Save: {save_path} (best {best:.4f})')
        else:
            patience += 1
            if patience >= EARLY_STOP_PATIENCE:
                print(f'Early stop at epoch {ep}')
                break

    total_time = time.perf_counter() - start_all
    print(f'[{variant_name}-F{fold_idx}] Done. Total {fmt(total_time)} | Best Val {best:.4f}')
    return best

print('Training loop defined.')

In [ ]:
# ===== Evaluation functions (from crossvalidation.ipynb) =====
from skimage.measure import EllipseModel, ransac

def dice_binary_np(pred_bin_255, gt_bin_255, smooth=1e-6):
    p = (pred_bin_255 > 0).astype(np.uint8)
    g = (gt_bin_255 > 0).astype(np.uint8)
    inter = (p & g).sum()
    union = p.sum() + g.sum()
    return (2*inter + smooth) / (union + smooth)

def mask_to_edge(mask_bin, thickness=3):
    contours, _ = cv2.findContours((mask_bin>0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    edge = np.zeros_like(mask_bin, dtype=np.uint8)
    for cnt in contours:
        cv2.drawContours(edge, [cnt], -1, 255, thickness=thickness)
    return edge

def ellipse_params_to_mask(params, H, W):
    cx = params[0]*W; cy = params[1]*H
    w  = params[2]*W; h  = params[3]*H
    angle = params[4]*180.0
    mask = np.zeros((H,W), dtype=np.uint8)
    center = (int(cx), int(cy))
    axes   = (max(1,int(w/2)), max(1,int(h/2)))
    cv2.ellipse(mask, center, axes, angle, 0, 360, 255, thickness=-1)
    return mask

def fit_ellipse_ransac(edge_points, min_samples=5, residual_threshold=2.0, max_trials=100):
    if len(edge_points) < 5:
        return None
    points = edge_points[:, ::-1].astype(np.float64)
    try:
        model, inliers = ransac(points, EllipseModel, min_samples=min_samples,
                                residual_threshold=residual_threshold, max_trials=max_trials)
        if inliers is not None and np.sum(inliers) >= 5:
            final_model = EllipseModel()
            final_model.estimate(points[inliers])
            xc, yc, a, b, theta = final_model.params
            return ((float(xc), float(yc)), (float(2*a), float(2*b)), float(np.degrees(theta)))
    except Exception:
        pass
    return None

def binary_to_ellipse_params(mask_bin, residual_threshold=2.0):
    ys, xs = np.where(mask_bin > 0)
    if len(xs) < 5:
        return None
    edge_points = np.column_stack([ys, xs])
    ellipse = fit_ellipse_ransac(edge_points, residual_threshold=residual_threshold, max_trials=100)
    if ellipse is None:
        return None
    (cx,cy), (w,h), angle = ellipse
    cx_n = np.clip(cx / mask_bin.shape[1], 0, 1)
    cy_n = np.clip(cy / mask_bin.shape[0], 0, 1)
    a_n  = np.clip(w  / mask_bin.shape[1], 1e-6, 1.0)
    b_n  = np.clip(h  / mask_bin.shape[0], 1e-6, 1.0)
    theta_n = (angle % 180) / 180.0
    return np.array([cx_n, cy_n, a_n, b_n, theta_n], dtype=np.float32)

def _max_contour_points_yx(mask_bin_255):
    m = (mask_bin_255 > 0).astype(np.uint8)
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return None
    cnt = max(contours, key=cv2.contourArea)
    pts = cnt.reshape(-1, 2)
    return pts[:, ::-1].astype(np.int32)

def ellipse_mask_from_fullmax_contour(vis_255, occ_255):
    full = ((vis_255 > 0) | (occ_255 > 0)).astype(np.uint8) * 255
    pts_yx = _max_contour_points_yx(full)
    if pts_yx is None or len(pts_yx) < 5:
        return np.zeros_like(full), None
    pts_xy = pts_yx[:, ::-1].astype(np.float32)
    try:
        ellipse = cv2.fitEllipse(pts_xy)
    except Exception:
        return np.zeros_like(full), None
    (cx, cy), (w, h), angle = ellipse
    H, W = full.shape
    mask = np.zeros((H, W), dtype=np.uint8)
    center = (int(round(cx)), int(round(cy)))
    axes = (max(1, int(round(w / 2))), max(1, int(round(h / 2))))
    cv2.ellipse(mask, center, axes, float(angle), 0, 360, 255, thickness=-1)
    return mask, ellipse

def ellipse_mask_from_visible_outer_arc(vis_255, occ_255, boundary_width=7,
                                        vis_near_radius=2, outer_thickness=2):
    H, W = vis_255.shape
    full = ((vis_255 > 0) | (occ_255 > 0)).astype(np.uint8)
    contours, _ = cv2.findContours(full, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return np.zeros((H, W), np.uint8), None
    cnt = max(contours, key=cv2.contourArea)
    edge_outer = np.zeros((H, W), np.uint8)
    cv2.drawContours(edge_outer, [cnt], -1, 255, thickness=outer_thickness)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    vis = (vis_255 > 0).astype(np.uint8)
    occ = (occ_255 > 0).astype(np.uint8)
    vis_d = cv2.dilate(vis, k, iterations=1)
    occ_d = cv2.dilate(occ, k, iterations=1)
    boundary = ((vis_d & occ_d) > 0).astype(np.uint8)
    boundary_d = cv2.dilate(boundary, k, iterations=max(1, boundary_width))
    allowed_outer = np.where(boundary_d > 0, 0, edge_outer)
    if vis_near_radius > 0:
        vis_near = cv2.dilate(vis, k, iterations=vis_near_radius)
    else:
        vis_near = vis
    edge_arc = np.where(vis_near > 0, allowed_outer, 0)
    pts_yx = np.column_stack(np.where(edge_arc > 0))
    if len(pts_yx) < 5:
        return np.zeros((H, W), np.uint8), None
    pts_xy = pts_yx[:, ::-1].astype(np.float32)
    try:
        ellipse = cv2.fitEllipse(pts_xy)
    except Exception:
        return np.zeros((H, W), np.uint8), None
    (cx, cy), (w, h), angle = ellipse
    mask = np.zeros((H, W), dtype=np.uint8)
    center = (int(round(cx)), int(round(cy)))
    axes = (max(1, int(round(w / 2))), max(1, int(round(h / 2))))
    cv2.ellipse(mask, center, axes, float(angle), 0, 360, 255, thickness=-1)
    return mask, ellipse


@torch.no_grad()
def evaluate_method3(model, val_loader, device):
    """Evaluate 6-class segmentation model with multiple ellipse fitting strategies."""
    model.eval()
    lid_scores, iris_scores, pupil_scores = [], [], []
    iris_scores_before_ellipse, pupil_scores_before_ellipse = [], []
    per_rows = []

    for batch in tqdm(val_loader, desc='Eval', leave=False):
        img = batch['image'].to(device)
        gt_lid   = batch['mask_lid'].cpu().numpy()
        gt_iris  = batch['mask_iris'].cpu().numpy()
        gt_pupil = batch['mask_pupil'].cpu().numpy()
        filenames = batch.get('filename', None)

        with autocast():
            out = model(img)
            logits = out['five_class_seg']

        pred_labels = torch.argmax(logits, dim=1).cpu().numpy()

        for b in range(pred_labels.shape[0]):
            pred = pred_labels[b]
            filename = filenames[b] if filenames is not None else str(b)

            # Eyelid: class 1 | 2 | 4
            lid_m3_bin = (((pred==1)|(pred==2)|(pred==4)).astype(np.uint8)*255)
            lid_d = float(dice_binary_np(lid_m3_bin, gt_lid[b]))
            lid_scores.append(lid_d)

            iris_vis = ((pred==2).astype(np.uint8)*255)
            iris_occ = ((pred==3).astype(np.uint8)*255)
            pupil_vis = ((pred==4).astype(np.uint8)*255)
            pupil_occ = ((pred==5).astype(np.uint8)*255)

            # raw
            iris_raw = (((pred==2)|(pred==3)).astype(np.uint8)*255)
            pupil_raw = (((pred==4)|(pred==5)).astype(np.uint8)*255)
            iris_raw_d = float(dice_binary_np(iris_raw, gt_iris[b]))
            pupil_raw_d = float(dice_binary_np(pupil_raw, gt_pupil[b]))

            iris_scores_before_ellipse.append(iris_raw_d)
            pupil_scores_before_ellipse.append(pupil_raw_d)

            # ransac_whole
            iris_edge = mask_to_edge(iris_raw, thickness=3)
            pupil_edge = mask_to_edge(pupil_raw, thickness=3)
            iris_params3 = binary_to_ellipse_params(iris_edge)
            pupil_params3 = binary_to_ellipse_params(pupil_edge)
            iris_ransac = ellipse_params_to_mask(iris_params3, IMAGE_HEIGHT, IMAGE_WIDTH) if iris_params3 is not None else np.zeros((IMAGE_HEIGHT,IMAGE_WIDTH), np.uint8)
            pupil_ransac = ellipse_params_to_mask(pupil_params3, IMAGE_HEIGHT, IMAGE_WIDTH) if pupil_params3 is not None else np.zeros((IMAGE_HEIGHT,IMAGE_WIDTH), np.uint8)
            iris_ransac_d = float(dice_binary_np(iris_ransac, gt_iris[b]))
            pupil_ransac_d = float(dice_binary_np(pupil_ransac, gt_pupil[b]))

            iris_scores.append(iris_ransac_d)
            pupil_scores.append(pupil_ransac_d)

            # fullmax
            iris_fullmax, _ = ellipse_mask_from_fullmax_contour(iris_vis, iris_occ)
            pupil_fullmax, _ = ellipse_mask_from_fullmax_contour(pupil_vis, pupil_occ)
            iris_fullmax_d = float(dice_binary_np(iris_fullmax, gt_iris[b]))
            pupil_fullmax_d = float(dice_binary_np(pupil_fullmax, gt_pupil[b]))

    result = {
        'lid': float(np.mean(lid_scores)) if lid_scores else 0.0,
        'iris': float(np.mean(iris_scores)) if iris_scores else 0.0,
        'pupil': float(np.mean(pupil_scores)) if pupil_scores else 0.0,
        'mean': float(np.mean([np.mean(lid_scores), np.mean(iris_scores), np.mean(pupil_scores)])) if lid_scores else 0.0,
        'iris_before_ellipse': float(np.mean(iris_scores_before_ellipse)) if iris_scores_before_ellipse else 0.0,
        'pupil_before_ellipse': float(np.mean(pupil_scores_before_ellipse)) if pupil_scores_before_ellipse else 0.0,
    }
    return result, per_rows

print('Evaluation functions defined.')

In [ ]:
# ===== Main Training Loop =====
import gc

for variant_name in LOSS_VARIANTS:
    print(f'\n{"="*80}')
    print(f'Training variant: {variant_name}')
    print(f'{"="*80}')

    for fold_idx in range(NUM_FOLDS):
        save_path = ABLATION_MODEL_DIR / f'{variant_name}_fold{fold_idx}_best.pth'
        if save_path.exists():
            print(f'  [SKIP] {save_path} already exists')
            continue

        print(f'\n--- {variant_name} Fold {fold_idx} ---')

        # Data
        train_indices = fold_indices[str(fold_idx)]['train']
        val_indices   = fold_indices[str(fold_idx)]['val']
        train_paths = [image_paths[i] for i in train_indices]
        val_paths   = [image_paths[i] for i in val_indices]

        train_ds = EyeSegmentationDataset(train_paths, LABEL_SEG_DIR, LABEL_OBB_DIR, transform=True)
        val_ds   = EyeSegmentationDataset(val_paths, LABEL_SEG_DIR, LABEL_OBB_DIR, transform=False)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
        val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                                  num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

        # Model
        model = UNetMethod4().to(device)
        criterion = create_criterion(variant_name).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        scaler = GradScaler()

        # Train
        run_ablation_fold(model, train_loader, val_loader, criterion, optimizer, scaler,
                          device, variant_name, fold_idx)

        # Cleanup
        del model, criterion, optimizer, scaler, train_ds, val_ds, train_loader, val_loader
        gc.collect()
        torch.cuda.empty_cache()

print('\nAll training complete.')

In [ ]:
# ===== Evaluation =====
result_dir = Path('results')
result_dir.mkdir(exist_ok=True)

ablation_results = {}

for variant_name in LOSS_VARIANTS:
    print(f'\nEvaluating: {variant_name}')
    fold_results = []

    for fold_idx in range(NUM_FOLDS):
        model_path = ABLATION_MODEL_DIR / f'{variant_name}_fold{fold_idx}_best.pth'
        if not model_path.exists():
            print(f'  WARNING: {model_path} not found')
            continue

        # Load model
        model = UNetMethod4().to(device)
        ckpt = torch.load(model_path, map_location=device)
        model.load_state_dict(ckpt['model'])

        # Data
        val_indices = fold_indices[str(fold_idx)]['val']
        val_paths = [image_paths[i] for i in val_indices]
        val_ds = EyeSegmentationDataset(val_paths, LABEL_SEG_DIR, LABEL_OBB_DIR, transform=False)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

        result, _ = evaluate_method3(model, val_loader, device)
        fold_results.append({
            'fold': fold_idx,
            'eyelid': result['lid'],
            'iris': result['iris'],
            'pupil': result['pupil'],
            'mean': result['mean'],
            'iris_before_ellipse': result['iris_before_ellipse'],
            'pupil_before_ellipse': result['pupil_before_ellipse'],
        })
        print(f'  Fold {fold_idx}: Eyelid={result["lid"]:.4f} Iris={result["iris"]:.4f} Pupil={result["pupil"]:.4f} Mean={result["mean"]:.4f}')

        del model
        gc.collect()
        torch.cuda.empty_cache()

    if fold_results:
        df_results = pd.DataFrame(fold_results)
        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        csv_path = result_dir / f'ablation_loss_{variant_name}_eval_{ts}.csv'
        df_results.to_csv(csv_path, index=False)
        print(f'  Saved: {csv_path}')
        ablation_results[variant_name] = df_results

print('\nEvaluation complete.')

In [ ]:
# ===== Load Baseline Method4 results =====
import glob

# Find the latest Method4 CSV
m4_csvs = sorted(glob.glob(str(result_dir / 'cv_eval_method4_*.csv')))
if m4_csvs:
    baseline_csv = m4_csvs[-1]
    print(f'Baseline CSV: {baseline_csv}')
    df_baseline = pd.read_csv(baseline_csv)
    print(df_baseline)
else:
    print('WARNING: No baseline Method4 CSV found. Using hardcoded values.')
    df_baseline = pd.DataFrame({
        'fold': [0, 1, 2, 3, 4],
        'eyelid': [0.9846, 0.9869, 0.9823, 0.9809, 0.9817],
        'iris': [0.9620, 0.9479, 0.9544, 0.9636, 0.9600],
        'pupil': [0.9247, 0.9057, 0.9115, 0.9081, 0.9278],
        'mean': [0.9571, 0.9469, 0.9494, 0.9509, 0.9565],
    })

ablation_results['baseline'] = df_baseline

In [ ]:
# ===== Comparison Table =====
comparison_rows = []

for name, df_r in ablation_results.items():
    row = {
        'variant': name,
        'eyelid_mean': df_r['eyelid'].mean(),
        'eyelid_std': df_r['eyelid'].std(),
        'iris_mean': df_r['iris'].mean(),
        'iris_std': df_r['iris'].std(),
        'pupil_mean': df_r['pupil'].mean(),
        'pupil_std': df_r['pupil'].std(),
        'mean_mean': df_r['mean'].mean(),
        'mean_std': df_r['mean'].std(),
    }
    comparison_rows.append(row)

df_comparison = pd.DataFrame(comparison_rows)

print('\n' + '=' * 100)
print('Loss Function Ablation: Comparison Table')
print('=' * 100)
for _, row in df_comparison.iterrows():
    print(f"{row['variant']:25s} | Eyelid {row['eyelid_mean']:.4f}+/-{row['eyelid_std']:.4f} | "
          f"Iris {row['iris_mean']:.4f}+/-{row['iris_std']:.4f} | "
          f"Pupil {row['pupil_mean']:.4f}+/-{row['pupil_std']:.4f} | "
          f"Mean {row['mean_mean']:.4f}+/-{row['mean_std']:.4f}")

# Save
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
csv_path = result_dir / f'ablation_loss_comparison_{ts}.csv'
df_comparison.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')

In [ ]:
# ===== Statistical Tests: paired t-test (n=5 folds) =====
print('\n' + '=' * 80)
print('Statistical Tests: each variant vs Baseline (paired t-test, n=5)')
print('=' * 80)

metrics = ['eyelid', 'iris', 'pupil', 'mean']
baseline_vals = {m: df_baseline[m].values for m in metrics}

for variant_name in LOSS_VARIANTS:
    if variant_name not in ablation_results:
        continue
    df_v = ablation_results[variant_name]
    print(f'\n--- {variant_name} vs Baseline ---')
    for m in metrics:
        v_vals = df_v[m].values
        b_vals = baseline_vals[m]
        diff = v_vals - b_vals
        t_stat, p_val = stats.ttest_rel(v_vals, b_vals)
        sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'n.s.'
        print(f'  {m:8s}: diff={diff.mean():+.4f} | t={t_stat:.3f} | p={p_val:.4f} | {sig}')

In [ ]:
# ===== Per-class Dice Analysis (6 classes) =====

def compute_perclass_dice(model, val_loader, device, num_classes=6):
    """Compute per-class Dice for 6-class segmentation."""
    model.eval()
    class_inter = np.zeros(num_classes)
    class_union = np.zeros(num_classes)

    with torch.no_grad():
        for batch in tqdm(val_loader, desc='PerClass', leave=False):
            img = batch['image'].to(device)
            gt = batch['gt_sixcls'].numpy()
            with autocast():
                out = model(img)
                logits = out['five_class_seg']
            pred = torch.argmax(logits, dim=1).cpu().numpy()
            for c in range(num_classes):
                p = (pred == c).astype(np.uint8)
                g = (gt == c).astype(np.uint8)
                class_inter[c] += (p & g).sum()
                class_union[c] += p.sum() + g.sum()

    dice = (2 * class_inter + 1e-6) / (class_union + 1e-6)
    return dice

# Compute per-class Dice for baseline and all variants
perclass_results = {}

# Baseline (from existing checkpoints)
BASELINE_MODEL_DIR = Path('model/cv_300ep')
baseline_perclass_folds = []
for fold_idx in range(NUM_FOLDS):
    model_path = BASELINE_MODEL_DIR / f'method4_fold{fold_idx}_best.pth'
    if not model_path.exists():
        print(f'Baseline model not found: {model_path}')
        continue
    model = UNetMethod4().to(device)
    ckpt = torch.load(model_path, map_location=device)
    model.load_state_dict(ckpt['model'])
    val_indices = fold_indices[str(fold_idx)]['val']
    val_paths = [image_paths[i] for i in val_indices]
    val_ds = EyeSegmentationDataset(val_paths, LABEL_SEG_DIR, LABEL_OBB_DIR, transform=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    dice = compute_perclass_dice(model, val_loader, device)
    baseline_perclass_folds.append(dice)
    del model; gc.collect(); torch.cuda.empty_cache()

if baseline_perclass_folds:
    perclass_results['baseline'] = np.mean(baseline_perclass_folds, axis=0)

# Ablation variants
for variant_name in LOSS_VARIANTS:
    variant_perclass_folds = []
    for fold_idx in range(NUM_FOLDS):
        model_path = ABLATION_MODEL_DIR / f'{variant_name}_fold{fold_idx}_best.pth'
        if not model_path.exists():
            continue
        model = UNetMethod4().to(device)
        ckpt = torch.load(model_path, map_location=device)
        model.load_state_dict(ckpt['model'])
        val_indices = fold_indices[str(fold_idx)]['val']
        val_paths = [image_paths[i] for i in val_indices]
        val_ds = EyeSegmentationDataset(val_paths, LABEL_SEG_DIR, LABEL_OBB_DIR, transform=False)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
        dice = compute_perclass_dice(model, val_loader, device)
        variant_perclass_folds.append(dice)
        del model; gc.collect(); torch.cuda.empty_cache()
    if variant_perclass_folds:
        perclass_results[variant_name] = np.mean(variant_perclass_folds, axis=0)

# Display
print('\n' + '=' * 80)
print('Per-class Dice (6 classes, averaged over 5 folds)')
print('=' * 80)
class_names = ['background', 'conjunctiva', 'iris_vis', 'iris_occ', 'pupil_vis', 'pupil_occ']
header = f'{"Variant":25s} | ' + ' | '.join(f'{n:12s}' for n in class_names)
print(header)
print('-' * len(header))
for name, dice in perclass_results.items():
    vals = ' | '.join(f'{d:12.4f}' for d in dice)
    print(f'{name:25s} | {vals}')

# Delta from baseline
if 'baseline' in perclass_results:
    print('\nDelta from Baseline:')
    print('-' * len(header))
    for name, dice in perclass_results.items():
        if name == 'baseline': continue
        delta = dice - perclass_results['baseline']
        vals = ' | '.join(f'{d:+12.4f}' for d in delta)
        print(f'{name:25s} | {vals}')

In [ ]:
# ===== Visualization =====
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

variant_labels = ['Baseline\n(CE+Dice)', 'WeightedCE\n+Dice', 'Focal\n+Dice']
variant_keys = ['baseline', 'weighted_ce_dice', 'focal_dice']
colors = ['#4C72B0', '#DD8452', '#55A868']

# Panel 1: Mean Dice bar chart
ax = axes[0]
means = []
stds = []
for key in variant_keys:
    if key in ablation_results:
        df_r = ablation_results[key]
        means.append(df_r['mean'].mean())
        stds.append(df_r['mean'].std())
    else:
        means.append(0)
        stds.append(0)
bars = ax.bar(variant_labels, means, yerr=stds, color=colors, capsize=5, edgecolor='black', linewidth=0.5)
ax.set_ylabel('Mean Dice')
ax.set_title('Mean Dice Comparison')
ax.set_ylim(0.93, 0.97)
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001, f'{m:.4f}',
            ha='center', va='bottom', fontsize=9)

# Panel 2: Per-structure bar chart
ax = axes[1]
x = np.arange(3)  # eyelid, iris, pupil
width = 0.25
structure_names = ['Eyelid', 'Iris', 'Pupil']
for i, (key, label, color) in enumerate(zip(variant_keys, variant_labels, colors)):
    if key in ablation_results:
        df_r = ablation_results[key]
        vals = [df_r['eyelid'].mean(), df_r['iris'].mean(), df_r['pupil'].mean()]
        errs = [df_r['eyelid'].std(), df_r['iris'].std(), df_r['pupil'].std()]
        ax.bar(x + i*width, vals, width, yerr=errs, label=label.replace('\n', ' '),
               color=color, capsize=3, edgecolor='black', linewidth=0.5)
ax.set_xticks(x + width)
ax.set_xticklabels(structure_names)
ax.set_ylabel('Dice')
ax.set_title('Per-Structure Dice')
ax.set_ylim(0.88, 1.0)
ax.legend(fontsize=8)

# Panel 3: 6-class Dice heatmap
ax = axes[2]
if perclass_results:
    heatmap_data = []
    heatmap_labels = []
    for key in variant_keys:
        if key in perclass_results:
            heatmap_data.append(perclass_results[key])
            heatmap_labels.append(key)
    if heatmap_data:
        heatmap_arr = np.array(heatmap_data)
        im = ax.imshow(heatmap_arr, cmap='YlOrRd', aspect='auto', vmin=0.5, vmax=1.0)
        ax.set_xticks(range(6))
        ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=8)
        ax.set_yticks(range(len(heatmap_labels)))
        ax.set_yticklabels(heatmap_labels, fontsize=9)
        ax.set_title('Per-Class Dice Heatmap')
        for i in range(len(heatmap_labels)):
            for j in range(6):
                ax.text(j, i, f'{heatmap_arr[i,j]:.3f}', ha='center', va='center', fontsize=8)
        plt.colorbar(im, ax=ax, shrink=0.8)
else:
    ax.text(0.5, 0.5, 'No per-class data', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
fig_path = result_dir / 'analysis5_loss_ablation.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved: {fig_path}')

## Discussion

### Expected outcomes

1. **Weighted CE + Dice**: May improve rare-class (pupil_occ, iris_occ) Dice at the expense of
   dominant-class precision. The normalized weights (mean=1.0) ensure loss magnitude compatibility.

2. **Focal Loss + Dice**: gamma=2.0 down-weights easy examples, focusing training on hard pixels
   (class boundaries, ambiguous regions). No explicit class rebalancing.

### Key metrics to watch

- **pupil_occ** (0.03% of pixels): The most extreme minority class. If weighted CE improves this,
  it validates the class-imbalance hypothesis.
- **iris_occ** (2.08%): Second most important minority class for clinical accuracy.
- **Overall Mean Dice**: Must not degrade significantly even if minority classes improve.

### Conclusion

This ablation study provides empirical evidence for or against explicit class-imbalance handling
in the loss function, directly addressing the reviewer concern about loss function design choices.